In [2]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
#Set Base Path
BASE = '/content/drive/MyDrive/Text Miners/Actual Work'
print('Connected:', BASE)

Connected: /content/drive/MyDrive/Text Miners/Actual Work


In [4]:
#Confirm connection
import os
print(os.listdir(BASE))

['data', 'pre-processing', 'features', 'task1-sentiments', 'task2-topics', 'task3-combination', 'dashboard', 'BeforeYouStart.docx']


In [5]:
#Install required libraries
!pip install vaderSentiment gensim nltk pandas numpy --quiet

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.5 MB/s eta 0:00:00


True

In [6]:
#Imports
import pandas as pd
import numpy as np
import os
from nltk.tokenize import sent_tokenize
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim.models import Word2Vec

#Load Data & Models
df = pd.read_csv(f'{BASE}/task2-topics/lda_topic_assignments.csv')
print('Data shape:', df.shape)
print(df[['listing_id', 'neighbourhood', 'comments']].head(3))

#Load your trained Word2Vec model from feature engineering
w2v_model = Word2Vec.load(f'{BASE}/features/word2vec.model')
print('Word2Vec vocabulary size:', len(w2v_model.wv))

Data shape: (380505, 8)
   listing_id neighbourhood                                           comments
0       27934   Ratchathewi  We stayed in the apartment for a week and we e...
1       27934   Ratchathewi  My girlfriend and I recently stayed in Nuttee'...
2       27934   Ratchathewi  I stayed for one month at the condo and was re...
Word2Vec vocabulary size: 28321


In [7]:
#Seed words per aspect - just 2-3 reliable words to anchor each aspect
aspect_seeds = {
    'cleanliness':   ['clean', 'dirty', 'smell'],
    'location':      ['location', 'nearby', 'mrt'],
    'communication': ['host', 'responsive', 'reply'],
    'check_in':      ['checkin', 'arrival', 'key'],
    'value':         ['value', 'price', 'worth'],
    'amenities':     ['wifi', 'kitchen', 'aircon']
}

#Automatically expand each aspect using Word2Vec similarity
aspect_keywords = {}

for aspect, seeds in aspect_seeds.items():
    keywords = set(seeds)  #start with seed words
    for seed in seeds:
        try:
            similar = w2v_model.wv.most_similar(seed, topn=20)
            keywords.update([word for word, score in similar if score > 0.6])
        except KeyError:
            print(f'  [{aspect}] seed word "{seed}" not in vocabulary, skipping')
    aspect_keywords[aspect] = list(keywords)

#Print expanded dictionary
print('=== Expanded Aspect Keyword Dictionary ===\n')
for aspect, keywords in aspect_keywords.items():
    print(f'{aspect} ({len(keywords)} keywords):')
    print(f'  {sorted(keywords)}\n')

=== Expanded Aspect Keyword Dictionary ===

cleanliness (43 keywords):
  ['cigarette', 'clean', 'damaged', 'damp', 'dirt', 'dirty', 'disgusting', 'dust', 'dusty', 'filthy', 'fragrance', 'greasy', 'grime', 'gross', 'messy', 'moldy', 'neat', 'oder', 'odor', 'odour', 'oily', 'pipe', 'reek', 'reeked', 'rusty', 'sewage', 'sewer', 'smell', 'smelled', 'smelling', 'smelly', 'smelt', 'spotless', 'stain', 'stained', 'stench', 'stink', 'stinky', 'stunk', 'torn', 'unclean', 'ventilation', 'wiped']

location (31 keywords):
  ['asok', 'brt', 'bst', 'btr', 'bts', 'chongnonsi', 'close', 'closeby', 'downstair', 'downstairs', 'ekkamai', 'location', 'lrt', 'metro', 'monorail', 'mrt', 'mtr', 'near', 'nearby', 'nut', 'onnut', 'pharmacy', 'position', 'ratchatewi', 'ratchathewi', 'skytrain', 'subway', 'surasak', 'surrounded', 'train', 'udomsuk']

communication (40 keywords):
  ['accommodating', 'accommodative', 'accomodating', 'answer', 'answerd', 'answering', 'approachable', 'attentive', 'communicative', 'c

In [8]:
#Remove noisy words after Word2Vec expansion
noise_checkin = {'credit', 'debit', 'card', 'swipe', 'chanced', 'frowned', 'stumbled', 'arrived', 'arrive', 'arriving', 'check', 'checked', 'checking'}
noise_value = {'changer', 'earning', 'exchanger', 'withdrawing'}
noise_amenities = {'fi', 'ac', 'fan', 'arctic', 'icy', 'paced', '5g', 'exhaust'}

aspect_keywords['check_in'] = [w for w in aspect_keywords['check_in'] if w not in noise_checkin]
aspect_keywords['value'] = [w for w in aspect_keywords['value'] if w not in noise_value]
aspect_keywords['amenities'] = [w for w in aspect_keywords['amenities'] if w not in noise_amenities]

print('Cleaned check_in:', sorted(aspect_keywords['check_in']))
print(f'  → {len(aspect_keywords["check_in"])} keywords remaining\n')

print('Cleaned value:', sorted(aspect_keywords['value']))
print(f'  → {len(aspect_keywords["value"])} keywords remaining\n')

print('Cleaned amenities:', sorted(aspect_keywords['amenities']))
print(f'  → {len(aspect_keywords["amenities"])} keywords remaining\n')

Cleaned check_in: ['arrival', 'checkin', 'checkout', 'chek', 'code', 'departure', 'explanatory', 'key', 'keycard', 'lockbox', 'mailbox', 'padlock', 'passcode', 'password', 'scan']
  → 15 keywords remaining

Cleaned value: ['justified', 'money', 'price', 'pricing', 'prize', 'rate', 'steal', 'value', 'valued', 'vaule', 'vlaue', 'worth', 'worthy']
  → 13 keywords remaining

Cleaned amenities: ['airco', 'aircon', 'aircond', 'aircondition', 'airconditioner', 'airconditioning', 'aircons', 'broadband', 'cooking', 'cooktop', 'cookware', 'crockery', 'cutlery', 'fridge', 'internet', 'kichen', 'kitchen', 'kitchenette', 'kitchenware', 'microwave', 'oven', 'pantry', 'refrigerator', 'shower', 'silverware', 'stovetop', 'wifi']
  → 27 keywords remaining



In [9]:
#Sentence Tokenization + Aspect Detection + VADER
analyzer = SentimentIntensityAnalyzer()

def detect_aspects(sentence, aspect_keywords):
    """Return list of aspects whose keywords appear in the sentence."""
    sentence_lower = sentence.lower()
    return [
        aspect for aspect, keywords in aspect_keywords.items()
        if any(kw in sentence_lower for kw in keywords)
    ]

def get_polarity(sentence):
    """Return polarity label and compound score for a sentence."""
    score = analyzer.polarity_scores(sentence)['compound']
    if score >= 0.05:
        return 'positive', score
    elif score <= -0.05:
        return 'negative', score
    else:
        return 'neutral', score

#Build flat table — one row per (listing_id, sentence, aspect)
print('Processing reviews...')
rows = []
skipped = 0

for _, review in df.iterrows():
    if pd.isna(review['comments']):
        skipped += 1
        continue

    sentences = sent_tokenize(str(review['comments']))

    for sentence in sentences:
        aspects = detect_aspects(sentence, aspect_keywords)
        if not aspects:
            continue  #no aspect keyword found, skip sentence

        polarity, score = get_polarity(sentence)
        if polarity == 'neutral':
            continue  #neutral excluded from NSS

        for aspect in aspects:
            rows.append({
                'listing_id':    review['listing_id'],
                'neighbourhood': review['neighbourhood'],
                'sentence':      sentence,
                'aspect':        aspect,
                'polarity':      polarity,
                'compound_score': score
            })

sentence_df = pd.DataFrame(rows)
print(f'Skipped {skipped} null reviews')
print(f'Total aspect-sentence pairs: {len(sentence_df)}')
print(sentence_df.head(10))

Processing reviews...
Skipped 0 null reviews
Total aspect-sentence pairs: 585420
   listing_id neighbourhood  \
0       27934   Ratchathewi   
1       27934   Ratchathewi   
2       27934   Ratchathewi   
3       27934   Ratchathewi   
4       27934   Ratchathewi   
5       27934   Ratchathewi   
6       27934   Ratchathewi   
7       27934   Ratchathewi   
8       27934   Ratchathewi   
9       27934   Ratchathewi   

                                            sentence         aspect  polarity  \
0  Nuttee is a very nice host, and she did her be...       location  positive   
1  Nuttee is a very nice host, and she did her be...  communication  positive   
2  It is a beautiful condo, with a great view, in...       location  positive   
3  Her assistant, Suchart, was great, meeting us ...    cleanliness  positive   
4  <br/>The condo is at the 19th floor, quiet, mo...    cleanliness  positive   
5  <br/>\n<br/>The communication between Nuttee, ...       location  positive   
6  The ass

In [10]:
#Quick Sanity Check
print('=== Polarity Distribution ===')
print(sentence_df['polarity'].value_counts())

print('\n=== Aspect Distribution ===')
print(sentence_df['aspect'].value_counts())

print('\n=== Sample rows per aspect ===')
for aspect in aspect_keywords.keys():
    sample = sentence_df[sentence_df['aspect'] == aspect].head(2)
    for _, row in sample.iterrows():
        print(f'\n[{aspect}] ({row["polarity"]}, {row["compound_score"]:.3f})')
        print(f'  "{row["sentence"]}"')

=== Polarity Distribution ===
polarity
positive    550658
negative     34762
Name: count, dtype: int64

=== Aspect Distribution ===
aspect
location         189679
communication    164965
cleanliness      134369
value             45032
amenities         37235
check_in          14140
Name: count, dtype: int64

=== Sample rows per aspect ===

[cleanliness] (positive, 0.625)
  "Her assistant, Suchart, was great, meeting us when we arrived, arranging cleaning of the condo, and seeing us off when we left."

[cleanliness] (positive, 0.778)
  "<br/>The condo is at the 19th floor, quiet, modern and clean, and you realy have a "great city view" by day and much more by night."

[location] (positive, 0.807)
  "Nuttee is a very nice host, and she did her best to accommodate us."

[location] (positive, 0.920)
  "It is a beautiful condo, with a great view, in a great location near Victory Monument."

[communication] (positive, 0.807)
  "Nuttee is a very nice host, and she did her best to accommodate 

In [11]:
#Count positive and negative sentences per listing per aspect
agg = (
    sentence_df
    .groupby(['listing_id', 'neighbourhood', 'aspect', 'polarity'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

#Ensure both columns exist even if one polarity never appears
for col in ['positive', 'negative']:
    if col not in agg.columns:
        agg[col] = 0

agg = agg.rename(columns={'positive': 'PA', 'negative': 'NA'})

#Compute NSS = (PA - NA) / (PA + NA)
agg['NSS'] = (agg['PA'] - agg['NA']) / (agg['PA'] + agg['NA'])

print('=== NSS ===')
print(agg.head(15))
print(f'\nTotal rows: {len(agg)}')

=== NSS ===
polarity  listing_id neighbourhood         aspect  NA  PA       NSS
0              27934   Ratchathewi      amenities   2   6  0.500000
1              27934   Ratchathewi       check_in   1   5  0.666667
2              27934   Ratchathewi    cleanliness   0  31  1.000000
3              27934   Ratchathewi  communication   0  32  1.000000
4              27934   Ratchathewi       location   3  97  0.940000
5              27934   Ratchathewi          value   0   9  1.000000
6              48736    Rat Burana    cleanliness   0   1  1.000000
7              55681      Bang Rak      amenities   1   2  0.333333
8              55681      Bang Rak       check_in   0   3  1.000000
9              55681      Bang Rak    cleanliness   0   6  1.000000
10             55681      Bang Rak  communication   0  13  1.000000
11             55681      Bang Rak       location   4  21  0.680000
12             55681      Bang Rak          value   0   7  1.000000
13            105042   Khlong Toei  

In [12]:
#Summary Stats
print('=== Mean NSS per Aspect (across all listings) ===')
print(agg.groupby('aspect')['NSS'].mean().sort_values(ascending=False).round(4))

print('\n=== NSS Distribution per Aspect ===')
print(agg.groupby('aspect')['NSS'].describe().round(4))

=== Mean NSS per Aspect (across all listings) ===
aspect
communication    0.8935
location         0.8853
value            0.8777
cleanliness      0.8517
amenities        0.6368
check_in         0.6130
Name: NSS, dtype: float64

=== NSS Distribution per Aspect ===
                 count    mean     std  min     25%  50%  75%  max
aspect                                                            
amenities       9353.0  0.6368  0.5951 -1.0  0.4286  1.0  1.0  1.0
check_in        5739.0  0.6130  0.6766 -1.0  0.3590  1.0  1.0  1.0
cleanliness    13726.0  0.8517  0.3684 -1.0  0.9000  1.0  1.0  1.0
communication  14454.0  0.8935  0.3136 -1.0  0.9707  1.0  1.0  1.0
location       14290.0  0.8853  0.2897 -1.0  0.8889  1.0  1.0  1.0
value           9954.0  0.8777  0.3658 -1.0  1.0000  1.0  1.0  1.0


In [13]:
#Export
agg.to_csv(f'{BASE}/task3-combination/nss_scores.csv', index=False)
sentence_df.to_csv(f'{BASE}/task3-combination/sentence_aspect_polarity.csv', index=False)

print('Saved:')
print(f'{BASE}/task3-combination/nss_scores.csv')
print(f'{BASE}/task3-combination/sentence_aspect_polarity.csv')

Saved:
/content/drive/MyDrive/Text Miners/Actual Work/task3-combination/nss_scores.csv
/content/drive/MyDrive/Text Miners/Actual Work/task3-combination/sentence_aspect_polarity.csv


In [ ]:
# =============================================================
# TASK 3 RULE-BASED SENTIMENT ANALYSIS — RESULTS ANALYSIS
# =============================================================
#
# METHODOLOGY SUMMARY
# - Keyword dictionary built using Word2Vec expansion from seed words, then manually pruned to remove noise (e.g. 'credit', 'debit' from check_in; 'arctic', 'icy' from amenities)
# - Sentences tokenized using NLTK sent_tokenize
# - Aspect detected via keyword substring matching (lowercased)
# - Polarity assigned using VADER compound score:
#     positive  : compound >= 0.05
#     negative  : compound <= -0.05
#     neutral   : excluded from NSS computation - neutral sentences carry no directional signal (neither praise nor complaint) and have no place in the NSS formula
#                 (PA - NA) / (PA + NA), which is defined only over sentences with clear polarity. Including neutral sentences would require padding the denominator
#                 with non-informative counts, diluting the score and making NSS harder to interpret meaningfully.
# - NSS formula: (PA - NA) / (PA + NA) per listing per aspect
#
# KEYWORD DICTIONARY SIZES (after pruning)
# - cleanliness   : 43 keywords
# - location      : 31 keywords
# - communication : 40 keywords
# - check_in      : 15 keywords
# - value         : 13 keywords
# - amenities     : 27 keywords
#
# COVERAGE STATS
# - Total reviews processed       : 380,505
# - Null reviews skipped          : 0
# - Total aspect-sentence pairs   : 585,420
# - Total listing-aspect NSS rows : 67,516
#
# ASPECT SENTENCE DISTRIBUTION
# - location         : 189,679 sentences
# - communication    : 164,965 sentences
# - cleanliness      : 134,369 sentences
# - value            :  45,032 sentences
# - amenities        :  37,235 sentences
# - check_in         :  14,140 sentences
#
# POLARITY DISTRIBUTION (across all aspect-sentence pairs)
# - positive : 550,658
# - negative :  34,762
# Ratio ~94% positive, consistent with overall dataset skew
#
# NSS RESULTS (mean across all listings, descending)
# ┌────────────────┬──────────┐
# │ Aspect         │ Mean NSS │
# ├────────────────┼──────────┤
# │ communication  │  0.8935  │
# │ location       │  0.8853  │
# │ value          │  0.8777  │
# │ cleanliness    │  0.8517  │
# │ amenities      │  0.6368  │
# │ check_in       │  0.6130  │
# └────────────────┴──────────┘
#
# KEY FINDINGS
# 1. Communication and location are the highest-rated aspects.
#    Guests are consistently satisfied with Bangkok Airbnb hosts and property locations (proximity to BTS/MRT, amenities).
#
# 2. Amenities and check-in score notably lower than other aspects.
#    This suggests guests have more mixed experiences with physical facilities and the arrival/key handover process specifically.
#    These are actionable areas for hosts to improve.
#
# 3. Check-in has the highest variance (std = 0.68), meaning experiences are highly inconsistent across listings -
#    some listings have seamless check-in (NSS = 1.0), others have frequent complaints (NSS = -1.0).
#
# 4. Cleanliness has a bimodal pattern - most listings score NSS = 1.0 (75th percentile = 1.0) but the mean is dragged
#    down by a minority of listings with serious cleanliness issues.
#    These outlier listings are important to surface in the dashboard.
#
# KNOWN LIMITATIONS
# 1. VADER was designed for social media text. It occasionally misclassifies sentences where a negative word appears in a
#    positive context (e.g. "sorted out a minor problem with the key" scored negative despite describing a positive resolution).
#
# 2. Keyword matching is substring-based, not semantic. A sentence can match multiple aspects simultaneously (e.g. "the host showed
#    us the wifi" triggers both communication and amenities), which inflates sentence counts but is broadly acceptable.
#
# 3. Neutral sentences (|compound| < 0.05) are intentionally excluded from NSS. They carry no directional sentiment signal and are
#    incompatible with the (PA - NA) / (PA + NA) formula. However, listings whose guests write consistently mild/measured reviews
#    may have fewer sentences crossing the ±0.05 threshold, resulting in NSS scores computed from a smaller sample - potentially less
#    stable than listings with more expressive reviewers.